# UCSF-PDGM: deep-learning only (fine-tuned MobileNetV3 / EfficientNet-B0 / ResNet18), no traditional classifiers

**What this notebook does**
- Each of the 4 MRI slices (T1, T1c, T2, FLAIR) goes through ONE shared ImageNet-pretrained CNN that is **fine-tuned** end to end. The 4 feature vectors are concatenated.
- Age + sex go through a small branch. During training they are hidden for 30% of patients (modality dropout), so the same network learns to work with and without them.
- Class imbalance: class-weighted loss on the training part only (or a balanced sampler). Test folds are never balanced.
- Early stopping uses a validation part inside each training fold (never the test fold).
- Parts: MRI only, MRI + clinical, and MRI + clinical with age/sex hidden at test time (0/25/50/75/100%, only age, only sex), for binary (grade 2/3 vs 4) and 3-class.

**How to run**: use a free GPU (Kaggle or Colab: Runtime > GPU). On a CPU laptop set `MODELS = ["mobilenet_v3_small"]`, `IMG = 128`, `N_FOLDS = 3`, one task at a time. This code was tested only on random synthetic data (it runs end to end); it has NOT been run on the real dataset, so check the first fold's output before leaving it running.

**Honest note**: fine-tuning can help, but with 495 patients, one slice each, and only 99 / 43 / 56 minority patients, results can also stay close to the earlier models. Report whatever comes out, for all models, and do not pick the best model by its test score.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
!pip install -q datasets torch torchvision scikit-learn pandas openpyxl

In [2]:
# ===== 1) SETTINGS (change things only here) =====
import os, copy, time, random, warnings
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (matthews_corrcoef, f1_score, balanced_accuracy_score, accuracy_score,
                             roc_auc_score, recall_score)
warnings.filterwarnings("ignore")

SYNTHETIC_TEST = False        # True = tiny random data, only to check that the code runs (no download)
SEEDS      = [42]             # use [42, 43, 44] on a GPU for the final run
N_FOLDS    = 5
IMG        = 160              # image side in pixels (smaller = faster). 224 is better but slower
EPOCHS     = 20               # maximum epochs, early stopping decides the real number
PATIENCE   = 5                # stop when validation macro-F1 has not improved for this many epochs
BATCH      = 16
LR_BACKBONE, LR_HEAD = 1e-4, 1e-3
MODELS     = ["mobilenet_v3_small", "efficientnet_b0", "resnet18"]   # add "mobilenet_v3_large", "swin_t" ...
TASKS      = ["binary", "3class"]
BALANCE    = "weighted_loss"  # "weighted_loss" | "sampler" | "none"   (never both; applied to TRAINING data only)
CLIN_DROP  = 0.3              # during training, hide age+sex for 30% of patients, so the model learns to work without them
MISSING_SCENARIOS = {"0% missing": 0.0, "25% missing": .25, "50% missing": .5, "75% missing": .75,
                     "100% missing": 1.0, "only age missing": "age", "only sex missing": "sex"}
PRETRAINED = not SYNTHETIC_TEST   # ImageNet weights (needs internet once). Backbones start pretrained, then are fine-tuned
TAG        = "deep_finetune"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [3]:
# ===== 2) DATA: 4 MRI slices + age + sex + grade, loaded once into memory =====
def to_gray(im): return np.asarray(im.convert("L").resize((IMG, IMG)), dtype=np.uint8)

if SYNTHETIC_TEST:
    rng = np.random.default_rng(0); N = 60
    grade = rng.choice([2, 3, 4], size=N, p=[.2, .15, .65])
    X = rng.integers(0, 255, size=(N, 4, IMG, IMG), dtype=np.uint8)
    age = rng.integers(20, 85, size=N).astype(float); sex = rng.integers(0, 2, size=N).astype(float)
else:
    from datasets import load_dataset
    ds = load_dataset("MedOtter/UCSF-PDGM", split="test")      # 'test' is only the default split name, not an official test set
    X = np.stack([[to_gray(r[s]) for s in ["T1", "T1c", "T2", "FLAIR"]] for r in ds])
    grade = np.array(ds["who_cns_grade"]).astype(int)
    age = np.array(ds["age_at_mri"], dtype=float)
    sex = np.array([1.0 if s == "M" else 0.0 for s in ds["sex"]])
print("images", X.shape, "| grades", dict(zip(*np.unique(grade, return_counts=True))))

def make_labels(task):
    return (grade == 4).astype(int) if task == "binary" else np.searchsorted([2, 3, 4], grade)


README.md:   0%|          | 0.00/6.17k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 38.0MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/495 [00:00<?, ? examples/s]

images (495, 4, 160, 160) | grades {np.int64(2): np.int64(56), np.int64(3): np.int64(43), np.int64(4): np.int64(396)}


In [4]:
# ===== 3) MODEL: shared pretrained backbone on each of the 4 sequences -> concatenate -> (+ age/sex branch) -> classifier =====
import torchvision.models as tvm
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def build_backbone(name):
    w = "DEFAULT" if PRETRAINED else None
    if name.startswith("mobilenet_v3"):
        m = getattr(tvm, name)(weights=w); m.classifier = nn.Identity()
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=w); m.classifier = nn.Identity()
    elif name == "resnet18":
        m = tvm.resnet18(weights=w); m.fc = nn.Identity()
    elif name == "swin_t":
        m = tvm.swin_t(weights=w); m.head = nn.Identity()
    else:
        raise ValueError(name)
    with torch.no_grad(): d = m.eval()(torch.zeros(2, 3, IMG, IMG)).shape[1]
    return m, d

class FusionNet(nn.Module):
    def __init__(self, name, n_classes, use_clinical):
        super().__init__()
        self.backbone, d = build_backbone(name); self.use_clinical = use_clinical
        self.mri = nn.Sequential(nn.Dropout(0.3), nn.Linear(4 * d, 128), nn.ReLU())
        self.clin = nn.Sequential(nn.Linear(4, 16), nn.ReLU())
        self.out = nn.Linear(128 + 16, n_classes)
    def forward(self, x, clin):                       # x: [B,4,H,W] uint8-scaled floats, clin: [B,4]
        B, S, H, W = x.shape
        z = x.reshape(B * S, 1, H, W).expand(-1, 3, -1, -1)
        z = (z - MEAN.to(x.device)) / STD.to(x.device)
        f = self.backbone(z).reshape(B, -1)
        c = self.clin(clin) if self.use_clinical else torch.zeros(B, 16, device=x.device)
        return self.out(torch.cat([self.mri(f), c], 1))

# clinical columns = [age (scaled), sex, age_missing, sex_missing]; hidden values are replaced by the training mean/mode
def clin_matrix(idx, stats, hide_age, hide_sex):
    a = np.where(hide_age, stats["age_mean"], age[idx]); s = np.where(hide_sex, stats["sex_mode"], sex[idx])
    return np.stack([(a - stats["age_mean"]) / stats["age_std"], s, hide_age.astype(float), hide_sex.astype(float)], 1).astype(np.float32)

def augment(x):                                       # light augmentation, training only
    flip = torch.rand(x.shape[0], 1, 1, 1) < 0.5; x = torch.where(flip, x.flip(-1), x)
    x = x * (1 + 0.1 * torch.randn(x.shape[0], 1, 1, 1)) + 0.05 * torch.randn(x.shape[0], 1, 1, 1)
    dx, dy = np.random.randint(-8, 9, 2); return torch.roll(x, (int(dy), int(dx)), (2, 3))


In [5]:
# ===== 4) TRAIN ONE MODEL (class imbalance handled on training data only) + EVALUATE =====
def to_x(idx): return torch.tensor(X[idx], dtype=torch.float32) / 255.0

def predict(model, idx, clin):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(idx), 32):
            out.append(F.softmax(model(to_x(idx[i:i+32]).to(device), torch.tensor(clin[i:i+32]).to(device)), 1).cpu().numpy())
    return np.concatenate(out)

def score(y, proba, task):
    pred = proba.argmax(1); n = proba.shape[1]
    r = {"mcc": matthews_corrcoef(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
         "balanced_acc": balanced_accuracy_score(y, pred), "accuracy": accuracy_score(y, pred)}
    try: r["auc"] = roc_auc_score(y, proba[:, 1]) if n == 2 else roc_auc_score(y, proba, multi_class="ovr", average="macro")
    except ValueError: r["auc"] = np.nan
    for k, v in enumerate(recall_score(y, pred, average=None, labels=range(n), zero_division=0)): r[f"recall class {k}"] = v
    return r

def train_one(name, y, tr, use_clinical, seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_classes = int(y.max()) + 1
    tr_in, va = train_test_split(tr, test_size=0.15, stratify=y[tr], random_state=seed)   # inner validation for early stopping
    stats = dict(age_mean=age[tr_in].mean(), age_std=age[tr_in].std() + 1e-6, sex_mode=float(np.round(sex[tr_in].mean())))
    model = FusionNet(name, n_classes, use_clinical).to(device)
    counts = np.bincount(y[tr_in], minlength=n_classes)
    w = torch.tensor(len(tr_in) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w if BALANCE == "weighted_loss" else None)
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": list(model.mri.parameters()) + list(model.clin.parameters()) + list(model.out.parameters()), "lr": LR_HEAD}], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    sample_p = (1.0 / np.maximum(counts, 1))[y[tr_in]] if BALANCE == "sampler" else None
    best, best_state, bad = -1, None, 0
    va_clin = clin_matrix(va, stats, np.zeros(len(va), bool), np.zeros(len(va), bool))
    for ep in range(EPOCHS):
        model.train()
        order = np.random.choice(len(tr_in), len(tr_in), p=sample_p / sample_p.sum()) if sample_p is not None else np.random.permutation(len(tr_in))
        for i in range(0, len(order), BATCH):
            b = tr_in[order[i:i + BATCH]]
            if len(b) < 2: continue
            drop = np.random.rand(len(b)) < CLIN_DROP if use_clinical else np.zeros(len(b), bool)   # modality dropout
            xb = augment(to_x(b)).to(device); cb = torch.tensor(clin_matrix(b, stats, drop, drop)).to(device)
            opt.zero_grad(); loss = loss_fn(model(xb, cb), torch.tensor(y[b]).to(device)); loss.backward(); opt.step()
        sched.step()
        v = f1_score(y[va], predict(model, va, va_clin).argmax(1), average="macro")
        if v > best: best, best_state, bad = v, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, stats

def run_study():
    rows = []; t0 = time.time()
    for task in TASKS:
        y = make_labels(task)
        for name in MODELS:
            for mode, use_clin in [("MRI only", False), ("MRI + clinical", True)]:
                for seed in SEEDS:
                    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed)
                    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y)):
                        model, stats = train_one(name, y, tr, use_clin, seed)
                        scen = MISSING_SCENARIOS if use_clin else {"0% missing": 0.0}
                        for sc, spec in scen.items():
                            rng = np.random.default_rng(seed * 100 + fold)
                            if spec == "age":    ha, hs = np.ones(len(te), bool), np.zeros(len(te), bool)
                            elif spec == "sex":  ha, hs = np.zeros(len(te), bool), np.ones(len(te), bool)
                            else:                ha = hs = rng.random(len(te)) < spec
                            p = predict(model, te, clin_matrix(te, stats, ha, hs))
                            rows.append(dict(task=task, model=name, mode=mode, scenario=sc, seed=seed, fold=fold, **score(y[te], p, task)))
                        print(f"{task} | {name} | {mode} | seed {seed} fold {fold} done | {time.time()-t0:.0f}s", flush=True)
    return pd.DataFrame(rows)


In [6]:
# ===== 5) RUN, SUMMARISE, SAVE =====
results = run_study()
results.to_csv(f"deep_raw_{TAG}.csv", index=False)
cols = ["mcc", "macro_f1", "balanced_acc", "auc", "accuracy"]
g = results.groupby(["task", "model", "mode", "scenario"])[cols]
summary = (g.mean().round(3).astype(str) + " ± " + g.std().round(3).astype(str))
summary.to_csv(f"deep_summary_{TAG}.csv")
try: summary.to_excel(f"deep_summary_{TAG}.xlsx")
except Exception as e: print("excel not saved:", e)
for task in TASKS:
    print(f"\n### {task}: full data (0% missing)")
    print(summary.xs(task, level="task").xs("0% missing", level="scenario"))
    print(f"\n### {task}: MRI + clinical when clinical data are hidden")
    print(summary.xs(task, level="task").xs("MRI + clinical", level="mode")["macro_f1" if task == "3class" else "mcc"].unstack("scenario"))


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 113MB/s]


binary | mobilenet_v3_small | MRI only | seed 42 fold 0 done | 16s
binary | mobilenet_v3_small | MRI only | seed 42 fold 1 done | 29s
binary | mobilenet_v3_small | MRI only | seed 42 fold 2 done | 46s
binary | mobilenet_v3_small | MRI only | seed 42 fold 3 done | 63s
binary | mobilenet_v3_small | MRI only | seed 42 fold 4 done | 80s
binary | mobilenet_v3_small | MRI + clinical | seed 42 fold 0 done | 87s
binary | mobilenet_v3_small | MRI + clinical | seed 42 fold 1 done | 94s
binary | mobilenet_v3_small | MRI + clinical | seed 42 fold 2 done | 101s
binary | mobilenet_v3_small | MRI + clinical | seed 42 fold 3 done | 115s
binary | mobilenet_v3_small | MRI + clinical | seed 42 fold 4 done | 123s
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 184MB/s]


binary | efficientnet_b0 | MRI only | seed 42 fold 0 done | 190s
binary | efficientnet_b0 | MRI only | seed 42 fold 1 done | 215s
binary | efficientnet_b0 | MRI only | seed 42 fold 2 done | 298s
binary | efficientnet_b0 | MRI only | seed 42 fold 3 done | 328s
binary | efficientnet_b0 | MRI only | seed 42 fold 4 done | 395s
binary | efficientnet_b0 | MRI + clinical | seed 42 fold 0 done | 447s
binary | efficientnet_b0 | MRI + clinical | seed 42 fold 1 done | 486s
binary | efficientnet_b0 | MRI + clinical | seed 42 fold 2 done | 542s
binary | efficientnet_b0 | MRI + clinical | seed 42 fold 3 done | 598s
binary | efficientnet_b0 | MRI + clinical | seed 42 fold 4 done | 670s
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 210MB/s]


binary | resnet18 | MRI only | seed 42 fold 0 done | 707s
binary | resnet18 | MRI only | seed 42 fold 1 done | 741s
binary | resnet18 | MRI only | seed 42 fold 2 done | 774s
binary | resnet18 | MRI only | seed 42 fold 3 done | 800s
binary | resnet18 | MRI only | seed 42 fold 4 done | 828s
binary | resnet18 | MRI + clinical | seed 42 fold 0 done | 863s
binary | resnet18 | MRI + clinical | seed 42 fold 1 done | 885s
binary | resnet18 | MRI + clinical | seed 42 fold 2 done | 923s
binary | resnet18 | MRI + clinical | seed 42 fold 3 done | 944s
binary | resnet18 | MRI + clinical | seed 42 fold 4 done | 974s
3class | mobilenet_v3_small | MRI only | seed 42 fold 0 done | 982s
3class | mobilenet_v3_small | MRI only | seed 42 fold 1 done | 989s
3class | mobilenet_v3_small | MRI only | seed 42 fold 2 done | 1000s
3class | mobilenet_v3_small | MRI only | seed 42 fold 3 done | 1013s
3class | mobilenet_v3_small | MRI only | seed 42 fold 4 done | 1031s
3class | mobilenet_v3_small | MRI + clinical | 